# 02. TRTR Baseline (Train Real, Test Real)

This notebook runs a patient-aware cross-validation benchmark on purified tumor data.

**Data Processing:**
- Loads tumor-only spectra (y_tumor=1 from H5)
- Crops to fingerprint region (700-1800 cm⁻¹)
- Caps each slide to max 500 spectra (prevents bias from slides with many correlated spectra)

**Training Strategy:**
- Tests 4 baseline models: Apollo (RF+SVM), Random Forest, XGBoost, 1D-CNN
- All models train on the same patient-capped data (29k spectra, 58 slides)
- 5-fold cross-validation with patient-aware splits
- Single training pass per model per fold (no multi-epoch resampling)


In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import joblib

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.data.augmentation import AugmentedDataGenerator
from src.models.baselines import ApolloPipeline

c:\Users\Andrei\anaconda3\envs\glioma\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [2]:
# ==============================================================================
# EXPERIMENT REGIMEN CONTROLLER
# ==============================================================================
# Options: 
#   "TRTR" : Train Real, Test Real (The Clean Baseline)
#   "TSTR" : Train Synthetic, Test Real (Requires running CVAE Notebook first)
#   "TATR" : Train Augmented, Test Real (Uses SMOTE/MixUp on the fly)
#   "TETR"   : Train Real + Synthetic, Test Real (Enriched: Real + CVAE)

REGIMEN = "TETR" 

print(f"🚀 INITIALIZING NOTEBOOK IN {REGIMEN} MODE")

🚀 INITIALIZING NOTEBOOK IN TETR MODE


In [3]:
import os
import sys
from pathlib import Path

# Add src to path and fix working directory
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment

# Setup experiment from config
config_name = 'exp_104'
config, output_dir = setup_experiment(config_name)

print(f"✓ Experiment loaded: {config_name}")
print(f"✓ Output directory: {output_dir}")


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_104
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Experiment loaded: exp_104
✓ Output directory: results\results


In [4]:
# --- Load configuration from exp_103.yaml ---
DATA_PATH = config['paths']['processed_file']
SPLIT_FILE = config['paths']['splits_file']

# ==============================================================================
# EXPERIMENT REGIMEN CONTROLLER
# ==============================================================================
# Options: 
#   "TRTR"   : Train Real, Test Real (The Clean Baseline)
#   "TSTR"   : Train Synthetic, Test Real (CVAE Only)
#   "TATR"   : Train Augmented, Test Real (Classical SMOTE/MixUp)
#   "TETR"   : Train Real + Synthetic, Test Real (Enriched: Real + CVAE)

# Determine the correct output directory
if REGIMEN == "TRTR":
    MODEL_DIR = Path(config['paths']['models_baseline_dir'])
elif REGIMEN == "TSTR":
    MODEL_DIR = Path(config['paths']['models_tstr_dir'])
elif REGIMEN == "TATR":
    MODEL_DIR = Path(config['paths'].get('models_tatr_dir', 'results/models/tatr'))
elif REGIMEN == "TETR":
    # Fallback to a default 'tetr' folder if not explicitly in your YAML yet
    MODEL_DIR = Path(config['paths'].get('models_tetr_dir', f"results/{config_name}/models/tetr"))
    
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Splits file: {SPLIT_FILE}")
print(f"Model output dir: {MODEL_DIR}")

# Create model directory if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

print("Initializing Loader from config...")
loader = GliomaDataLoader.from_config(
    h5_data_path=DATA_PATH,
    config=config,
    memory_mapped=False
)

print(f"✓ Loader initialized. Total spectra: {loader.X.shape[0]}")   
print(f"✓ Number of folds: {len(loader)}")

Data path: results/exp_102/data/glioma_dataset.h5
Splits file: data/splits/strict_splits.json
Model output dir: results\exp_104\models\tetr
Initializing Loader from config...
Loading CV splits from: data/splits/strict_splits.json
Loading data from results\exp_102\data\glioma_dataset.h5...
[*] 'tumor_only=True': Filtering out healthy tissue...
    Filtered to 226284 Pure Tumor Spectra.
[OK] Spectral Crop: [700, 1800] cm-1 -> indices [337, 908]
  Wavenumber mapping: 699.91 -> 1799.14 cm-1
  Renormalizing to cropped region...
    After renormalization: min=0.000000, max=1.000000 (fingerprint region)
  Spectra shape after crop: (226284, 571)
✓ Loader initialized. Total spectra: 226284
✓ Number of folds: 5


In [5]:
# --- Reproducibility: Set seed from config ---
import random
import os

# Suppress TensorFlow warnings (imported only when CNN starts)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

RANDOM_SEED = config['training']['random_seed']

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print(f"Random seed set to: {RANDOM_SEED}")


Random seed set to: 42


In [6]:
# PRE-FLIGHT CHECK
print("="*60)
print(f"num_classes: {config['models']['cnn']['num_classes']}")
print(f"Binary mapping: {config['models']['cnn']['num_classes'] == 2}")
first_fold = next(iter(loader))
y_check = first_fold['train'][1].copy()
if np.min(y_check) == 1: y_check -= 1
print(f"Train label range: {np.unique(y_check)}")
print(f"Label counts: {np.bincount(y_check.astype(int))}")
if REGIMEN in ["TSTR", "TETR"]:
    synth_dir = Path(config['paths']['synthetic_dir'])
    synth_file = synth_dir / f"{first_fold['fold_name']}_X_synth.npy"
    print(f"Synthetic data exists: {synth_file.exists()}")
print("="*60)

num_classes: 6
Binary mapping: False

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples
Train label range: [0 1 2 3 4 5]
Label counts: [10563 33461 38518 30244 38708  8590]
Synthetic data exists: True


In [ ]:
import time
import os
import gc
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from src.models.baselines import ApolloPipeline

# Extract model configs
apollo_cfg = config['models']['apollo']
rf_cfg = config['models']['random_forest']
xgb_cfg = config['models']['xgboost']
cnn_cfg = config['models']['cnn']
num_classes = config['models']['cnn']['num_classes']
n_folds = config['cv_strategy']['n_folds']

benchmark_results = {
    'Apollo': [],
    'RF_Standard': [],
    'XGBoost': [],
    '1D_CNN': []
}


print(" Starting Multi-Model TRTR Benchmark (Patient-Capped Data)")
print(f"Data: {loader.X.shape[0]} tumor spectra, {loader.X.shape[1]} features")
print(f"Slides: {len(np.unique(loader.groups))}, Classes: {len(np.unique(loader.y_class))}")
print(f"Model output dir: {MODEL_DIR}\n")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*50}\nProcessing {fold_name} ({REGIMEN})\n{'='*50}")
    
    # 1. Validation Data ALWAYS comes from the real held-out fold
    # 1. Extract Data
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1].copy()
    X_train_raw = fold_data['train'][0].astype(np.float32)
    y_train_raw = fold_data['train'][1].copy()

    # Fix 1-indexed to 0-indexed if necessary
    if np.min(y_train_raw) == 1: y_train_raw -= 1
    if np.min(y_val) == 1: y_val -= 1
        
    # 2. Binary Mapping (DO THIS FIRST so augmentation sees the binary classes)
    if config['models']['cnn']['num_classes'] == 2:
        y_train_raw = np.where(y_train_raw < 3, 0, 1)
        y_val = np.where(y_val < 3, 0, 1)
    
    # 3. THE REGIMEN SWITCHBOARD
    if REGIMEN == "TRTR":
        X_train, y_train = X_train_raw, y_train_raw
        print(f"  ✓ Loaded REAL training data: {X_train.shape}")
        
    elif REGIMEN == "TSTR":
        synth_dir = Path(config['paths']['synthetic_dir'])
        X_train = np.load(synth_dir / f"{fold_name}_X_synth.npy").astype(np.float32)
        y_train = np.load(synth_dir / f"{fold_name}_y_synth.npy").copy()
        if X_train.ndim == 3:
            X_train = X_train.reshape(X_train.shape[0], -1)
        
        valid_indices = ~np.isnan(X_train).any(axis=1)
        X_train = X_train[valid_indices]
        y_train = y_train[valid_indices]
        
        n_removed = (~valid_indices).sum()
        if n_removed > 0:
            print(f"  ⚠ Removed {n_removed} samples with NaN values")
        print(f"  ✓ Loaded SYNTHETIC training data: {X_train.shape}")
        
    elif REGIMEN == "TATR":
        print("  ! Applying On-the-Fly TATR Augmentation...")
        augmentation_config = config.get('augmentation', {})

        if augmentation_config.get('enabled', False):
            aug_gen = AugmentedDataGenerator(X_train_raw, y_train_raw, augmentation_config)
            augmented_batches = list(aug_gen.get_batches(batch_size=256, epochs=1, shuffle=True))
            
            X_aug = np.vstack([batch_x for batch_x, _ in augmented_batches])
            y_aug = np.concatenate([batch_y for _, batch_y in augmented_batches])

            # Combine
            X_combined = np.vstack([X_train_raw, X_aug])
            y_combined = np.concatenate([y_train_raw, y_aug])
            
            # Final Shuffle
            shuffle_idx = np.random.permutation(len(y_combined))
            X_train = X_combined[shuffle_idx]
            y_train = y_combined[shuffle_idx]
            
            print(f"  ✓ Generated AUGMENTED training data: {X_train.shape}")
        else:
            X_train, y_train = X_train_raw, y_train_raw
            print("  ! Augmentation disabled: using real data only")
    elif REGIMEN == "TETR":
        print("  ! Combining REAL + SYNTHETIC data for TETR training...")
        synth_dir = Path(config['paths']['synthetic_dir'])
        X_synth = np.load(synth_dir / f"{fold_name}_X_synth.npy").astype(np.float32)
        y_synth = np.load(synth_dir / f"{fold_name}_y_synth.npy").copy()
        
        if X_synth.ndim == 3:
            X_synth = X_synth.reshape(X_synth.shape[0], -1)
        
        # Combine real + synthetic
        X_combined = np.vstack([X_train_raw, X_synth])
        y_combined = np.concatenate([y_train_raw, y_synth])
        
        # Shuffle
        shuffle_idx = np.random.permutation(len(y_combined))
        X_train = X_combined[shuffle_idx]
        y_train = y_combined[shuffle_idx]
        
        print(f"  ✓ Combined REAL + SYNTHETIC training data: {X_train.shape}")        
    else:
        raise ValueError(f"Unknown REGIMEN: {REGIMEN}")
    
    # Fix 1-indexed labels to 0-indexed if necessary
    if np.min(y_train) == 1:
        y_train = y_train - 1
    if np.min(y_val) == 1:
        y_val = y_val - 1
        
    
    print(f"Train: {X_train.shape[0]} spectra, Val: {X_val.shape[0]} spectra")
    
    # --- MODEL 1: Apollo Pipeline ---
    print(f"[1/4] Training Apollo (n_features={apollo_cfg['n_features']})...")
    apollo_pipeline = ApolloPipeline(
        n_features=apollo_cfg['n_features'],
        random_state=apollo_cfg['random_state']
    )
    apollo_pipeline.fit(X_train, y_train)
    y_pred_apollo = apollo_pipeline.predict(X_val)
    apollo_acc = accuracy_score(y_val, y_pred_apollo)
    benchmark_results['Apollo'].append(apollo_acc)
    joblib.dump(apollo_pipeline, f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    print(f"      ✓ Val Accuracy: {apollo_acc:.4f}")
    del y_pred_apollo, apollo_pipeline
    gc.collect()
    
    # --- MODEL 2: Random Forest ---
    print(f"[2/4] Training RF (n_est={rf_cfg['n_estimators']})...")
    rf = RandomForestClassifier(
        n_estimators=rf_cfg['n_estimators'],
        class_weight=rf_cfg['class_weight'],
        random_state=rf_cfg['random_state'],
        n_jobs=rf_cfg['n_jobs']
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_val)
    rf_acc = accuracy_score(y_val, y_pred_rf)
    benchmark_results['RF_Standard'].append(rf_acc)
    joblib.dump(rf, f"{MODEL_DIR}/rf_{fold_name}.joblib")
    print(f"      ✓ Val Accuracy: {rf_acc:.4f}")
    del rf, y_pred_rf
    gc.collect()
    
    # --- MODEL 3: XGBoost ---
    print(f"[3/4] Training XGBoost...")
    xgb_model = XGBClassifier(
    objective=xgb_cfg['objective'],
    num_class=xgb_cfg.get('num_class', None),  # Use None if key doesn't exist
    random_state=xgb_cfg['random_state'],
    n_jobs=xgb_cfg['n_jobs'],
    tree_method=xgb_cfg['tree_method']
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_val)
    xgb_acc = accuracy_score(y_val, y_pred_xgb)
    benchmark_results['XGBoost'].append(xgb_acc)
    xgb_model.save_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    print(f"      ✓ Val Accuracy: {xgb_acc:.4f}")
    del xgb_model, y_pred_xgb
    gc.collect()
    
    # --- MODEL 4: 1D-CNN ---
    print(f"[4/4] Training 1D-CNN (epochs={cnn_cfg['epochs']})...")
    try:
        # Import TensorFlow only when CNN training starts (avoids Windows DLL conflicts)
        import tensorflow as tf
        from src.models.cnn import build_1d_cnn
        
        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_SEED)
        
        # Reshape for CNN - use ACTUAL feature dimension from data (after cropping)
        num_features = X_train.shape[1]
        X_train_cnn = X_train.reshape(-1, num_features, 1)
        X_val_cnn = X_val.reshape(-1, num_features, 1)
        y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=cnn_cfg['num_classes'])
        y_val_cnn = tf.keras.utils.to_categorical(y_val, num_classes=cnn_cfg['num_classes'])
        
        # Build model with DYNAMIC input shape based on actual feature dimension
        cnn_model = build_1d_cnn(
            input_shape=(num_features, 1),
            num_classes=cnn_cfg['num_classes']
        )
        cnn_model.compile(
            optimizer=cnn_cfg['optimizer'],
            loss=cnn_cfg['loss'],
            metrics=['accuracy']
        )
        
        # Early stopping
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=cnn_cfg['early_stopping_patience'],
            restore_best_weights=True
        )
        
        # Train
        history = cnn_model.fit(
            X_train_cnn, y_train_cnn,
            validation_data=(X_val_cnn, y_val_cnn),
            epochs=cnn_cfg['epochs'],
            batch_size=cnn_cfg['batch_size'],
            callbacks=[early_stop],
            verbose=0
        )
        
        y_pred_cnn_probs = cnn_model.predict(X_val_cnn, verbose=0)
        y_pred_cnn = np.argmax(y_pred_cnn_probs, axis=1)
        cnn_acc = accuracy_score(y_val, y_pred_cnn)
        benchmark_results['1D_CNN'].append(cnn_acc)
        cnn_model.save(f"{MODEL_DIR}/cnn_{fold_name}.keras")
        print(f"      ✓ Val Accuracy: {cnn_acc:.4f}")
        print(f"      ✓ Trained epochs: {len(history.history['loss'])}")
        del cnn_model, X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn, y_pred_cnn_probs, y_pred_cnn
        tf.keras.backend.clear_session()
        gc.collect()
    except Exception as e:
        print(f"      ✗ CNN training failed: {str(e)[:80]}")
        benchmark_results['1D_CNN'].append(np.nan)

# --- Print Summary ---
print("\n" + "="*60)
print("🎉 TRTR BENCHMARK COMPLETE")
print("="*60)
print(f"\nData: Patient-capped to 500 spectra per slide")
print(f"All models trained on the same capped dataset\n")

for model_name, accuracies in benchmark_results.items():
    valid_accs = [acc for acc in accuracies if not np.isnan(acc)]
    if valid_accs:
        mean_acc = np.mean(valid_accs)
        std_acc = np.std(valid_accs)
        n_folds = len(valid_accs)
        print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f} ({n_folds} folds)")
    else:
        print(f"{model_name:>15}: SKIPPED")

print("="*60)


 Starting Multi-Model TRTR Benchmark (Patient-Capped Data)
Data: 226284 tumor spectra, 571 features
Slides: 58, Classes: 6
Model output dir: results\exp_104\models\tetr


 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples

Processing fold_0 (TETR)
  ! Combining REAL + SYNTHETIC data for TETR training...
  ✓ Combined REAL + SYNTHETIC training data: (280084, 571)
Train: 280084 spectra, Val: 45936 spectra
[1/4] Training Apollo (n_features=20)...
  [Apollo] RF Config: n_estimators=300, max_depth=10, n_features_selected=20
  [Apollo] SVM Config: C=1.0, kernel=rbf
  [Apollo] Fitting RF Selector on (280084, 571) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
      ✓ Val Accuracy: 0.7768
[2/4] Train

In [5]:
%reset -f

In [7]:
# import numpy as np
# import joblib
# import xgboost as xgb
# import tensorflow as tf
# from sklearn.metrics import accuracy_score
# import gc

# # Use MODEL_DIR from config (already set in previous cell)

# six_class_benchmark_results = {
#     'Apollo': [],
#     'RF_Standard': [],
#     'XGBoost': [],
#     '1D_CNN': []
# }

# binary_benchmark_results = {
#     'Apollo (Mapped to Binary)': [],
#     'RF_Standard (Mapped to Binary)': [],
#     'XGBoost (Mapped to Binary)': [],
#     '1D_CNN (Mapped to Binary)': []
# }

# print("🔍 Testing 6-Class Models on the Binary Task...")

# for fold_data in loader:
#     fold_name = fold_data['fold_name']
#     print(f"🔄 Evaluating {fold_name}...")
    
#     # Load validation data
#     X_val = fold_data['val'][0].astype(np.float32)
#     y_val_6_class = fold_data['val'][1]
    
#     # Fix 1-indexed to 0-indexed if necessary
#     if np.min(y_val_6_class) == 1:
#         y_val_6_class = y_val_6_class - 1
        
#     # Map True Labels to Binary: 0,1,2 → 0 (Mutant) | 3,4,5 → 1 (Wildtype)
#     y_val_binary = np.where(y_val_6_class < 3, 0, 1)
    
#     # --- 1. Apollo ---
#     apollo = joblib.load(f"{MODEL_DIR}/apollo_{fold_name}.joblib")
#     y_pred_apollo_6 = apollo.predict(X_val)
#     y_pred_apollo_bin = np.where(y_pred_apollo_6 < 3, 0, 1)
#     six_class_benchmark_results['Apollo'].append(accuracy_score(y_val_6_class, y_pred_apollo_6))
#     binary_benchmark_results['Apollo (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_apollo_bin))
    
#     # --- 2. Random Forest ---
#     rf = joblib.load(f"{MODEL_DIR}/rf_{fold_name}.joblib")
#     y_pred_rf_6 = rf.predict(X_val)
#     y_pred_rf_bin = np.where(y_pred_rf_6 < 3, 0, 1)
#     six_class_benchmark_results['RF_Standard'].append(accuracy_score(y_val_6_class, y_pred_rf_6))
#     binary_benchmark_results['RF_Standard (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_rf_bin))
    
#     # --- 3. XGBoost ---
#     xgb_model = xgb.XGBClassifier()
#     xgb_model.load_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
#     y_pred_xgb_6 = xgb_model.predict(X_val)
#     y_pred_xgb_bin = np.where(y_pred_xgb_6 < 3, 0, 1)
#     six_class_benchmark_results['XGBoost'].append(accuracy_score(y_val_6_class, y_pred_xgb_6))
#     binary_benchmark_results['XGBoost (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_xgb_bin))
    
#     # --- 4. 1D-CNN ---
#     cnn = tf.keras.models.load_model(f"{MODEL_DIR}/cnn_{fold_name}.keras")
#     X_val_cnn = X_val.reshape(-1, num_features, 1)
#     y_pred_cnn_probs = cnn.predict(X_val_cnn, verbose=0)
#     y_pred_cnn_6 = np.argmax(y_pred_cnn_probs, axis=1)
#     y_pred_cnn_bin = np.where(y_pred_cnn_6 < 3, 0, 1)
#     six_class_benchmark_results['1D_CNN'].append(accuracy_score(y_val_6_class, y_pred_cnn_6))
#     binary_benchmark_results['1D_CNN (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_cnn_bin))
    
#     # Cleanup memory
#     del apollo, rf, xgb_model, cnn

# # Print results
# print("\n=== 6-CLASS RESULTS (TRTR) ===")
# for model_name, accuracies in six_class_benchmark_results.items():
#     mean_acc = np.mean(accuracies)
#     std_acc = np.std(accuracies)
#     print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f}")

# print("\n=== BINARY RESULTS (Mapped from 6-Class Predictions) ===")
# for model_name, accuracies in binary_benchmark_results.items():
#     mean_acc = np.mean(accuracies)
#     std_acc = np.std(accuracies)
#     print(f"{model_name:>33}: {mean_acc:.4f} ± {std_acc:.4f}")